# M1 Environment Smoke Test

**Phase 1, Milestone M1 validation.** End-to-end smoke test of the Phase 1 Python research environment.

## Purpose

Per `docs/phase-1-detailed-plan.md` M1 validation:

> Able to create a test Jupyter notebook, run a polars operation on sample data, and commit the notebook to git.

This notebook satisfies that criterion. Real MNQ data loading is M2's responsibility; here we construct a small synthetic OHLCV sample inline so the smoke test is fully self-contained and reproducible without data dependencies.

## What this verifies

1. Cursor's Jupyter integration can run notebooks against the project `.venv` (selected via the `quant-research` kernel).
2. `polars` imports cleanly and key operations work end-to-end on Windows-native Python 3.13: `LazyFrame` construction, datetime parsing, group-by aggregation, and rolling window calculations.
3. The wider scientific stack (`numpy`, `scipy`, `matplotlib`) is importable.
4. The notebook can be saved and committed to git.

## What this does NOT verify

- Real MNQ data loading correctness (M2)
- Indicator correctness against reference libs (M3)
- Backtest engine logic (M4)

In [1]:
from datetime import datetime, timedelta

import numpy as np
import polars as pl

print(f"polars {pl.__version__}")
print(f"numpy  {np.__version__}")

polars 1.40.1
numpy  2.2.6


## Build a synthetic 1-minute OHLCV sample

Geometric Brownian motion to get something that looks vaguely market-like. Fixed seed for reproducibility.

In [2]:
rng = np.random.default_rng(seed=42)
n_bars = 390 * 5
start = datetime(2026, 4, 21, 9, 30)
timestamps = [start + timedelta(minutes=i) for i in range(n_bars)]

log_returns = rng.normal(loc=0.0, scale=0.0008, size=n_bars)
close = 18000.0 * np.exp(np.cumsum(log_returns))

high = close * (1.0 + np.abs(rng.normal(0, 0.0005, size=n_bars)))
low = close * (1.0 - np.abs(rng.normal(0, 0.0005, size=n_bars)))
open_ = np.concatenate([[close[0]], close[:-1]])
volume = rng.integers(low=50, high=2000, size=n_bars)

df = pl.DataFrame(
    {
        "timestamp": timestamps,
        "open": open_,
        "high": high,
        "low": low,
        "close": close,
        "volume": volume,
    }
)

print(f"Shape: {df.shape}")
df.head(5)

Shape: (1950, 6)


timestamp,open,high,low,close,volume
datetime[μs],f64,f64,f64,f64,i64
2026-04-21 09:30:00,18004.388461,18004.896289,17998.606197,18004.388461,1092
2026-04-21 09:31:00,18004.388461,17991.622309,17983.456039,17989.415268,1902
2026-04-21 09:32:00,17989.415268,18016.206432,17986.517659,18000.218653,1909
2026-04-21 09:33:00,18000.218653,18019.658217,18011.686919,18013.768047,992
2026-04-21 09:34:00,18013.768047,17987.609763,17971.121128,17985.673582,1309


## Exercise polars: per-day aggregation and rolling stats

Tests `LazyFrame` evaluation, datetime extraction, group-by, and rolling-window aggregation — the same primitives the M2 data pipeline will rely on.

In [3]:
daily = (
    df.lazy()
    .with_columns(date=pl.col("timestamp").dt.date())
    .group_by("date")
    .agg(
        bars=pl.len(),
        day_open=pl.col("open").first(),
        day_high=pl.col("high").max(),
        day_low=pl.col("low").min(),
        day_close=pl.col("close").last(),
        day_volume=pl.col("volume").sum(),
    )
    .sort("date")
    .collect()
)

daily

date,bars,day_open,day_high,day_low,day_close,day_volume
date,u32,f64,f64,f64,f64,i64
2026-04-21,870,18004.388461,18085.792098,17487.815701,17623.977602,869407
2026-04-22,1080,17623.977602,17732.317891,16497.311194,16528.448945,1089480


In [4]:
with_rolling = df.with_columns(
    [
        pl.col("close").rolling_mean(window_size=20).alias("sma_20"),
        pl.col("close").rolling_std(window_size=20).alias("std_20"),
    ]
)
with_rolling.tail(5)

timestamp,open,high,low,close,volume,sma_20,std_20
datetime[μs],f64,f64,f64,f64,i64,f64,f64
2026-04-22 17:55:00,16559.344636,16548.694526,16529.420694,16535.478557,745,16631.281681,40.722356
2026-04-22 17:56:00,16535.478557,16536.836157,16517.581538,16525.531678,701,16623.942509,45.842951
2026-04-22 17:57:00,16525.531678,16528.304338,16501.577188,16513.51732,392,16616.680722,51.225623
2026-04-22 17:58:00,16513.51732,16507.26359,16497.311194,16502.011857,1571,16608.266253,55.59011
2026-04-22 17:59:00,16502.011857,16531.102141,16528.145588,16528.448945,1173,16600.685103,55.620999


## Sanity checks

Hard assertions so a regression in the polars install or the env shows up as a notebook-level failure rather than silent corruption.

In [5]:
assert df.height == n_bars, f"row count mismatch: {df.height} vs {n_bars}"
assert df.schema["timestamp"] == pl.Datetime("us"), df.schema["timestamp"]
assert (df["high"] >= df["close"]).all()
assert (df["low"] <= df["close"]).all()
assert daily["bars"].sum() == n_bars
assert with_rolling.filter(pl.col("sma_20").is_not_null()).height == n_bars - 19

print("All checks passed.")

All checks passed.


## Result

Environment is functional. M1 validation criterion satisfied once this notebook is committed to git.

## Next step

Begin **M2: Data Pipeline** — read raw `MNQ_*.txt` files into polars with the schema specified in `docs/ai-project-instructions.md` Section 7.